In [1]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

# Step 1: Load standard dataset
categories = [
    'alt.atheism',
    'comp.graphics',
    'sci.space'
]

data = fetch_20newsgroups(
    subset='train',
    categories=categories,
    remove=('headers', 'footers', 'quotes')
)

documents = data.data
actual_labels = data.target

# Step 2: Convert documents into TF-IDF vectors
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

X = vectorizer.fit_transform(documents)

# Step 3: Apply K-Means clustering
k = 3

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X)

# Step 4: Map each cluster to the actual class
# Find the most common actual class in each cluster
cluster_to_class = {}

for cluster in range(k):
    indices = np.where(cluster_labels == cluster)[0]

    if len(indices) > 0:
        labels = actual_labels[indices]
        majority_class = np.bincount(labels).argmax()
        cluster_to_class[cluster] = majority_class

# Convert cluster labels to predicted class labels
predicted_labels = np.array([
    cluster_to_class[c] for c in cluster_labels
])

# Step 5: Calculate Purity
def calculate_purity(actual, predicted_clusters):
    total = len(actual)
    correct = 0

    for cluster in np.unique(predicted_clusters):
        indices = np.where(predicted_clusters == cluster)[0]
        labels = actual[indices]

        if len(labels) > 0:
            correct += np.bincount(labels).max()

    return correct / total

purity = calculate_purity(actual_labels, cluster_labels)

# Step 6: Calculate Precision, Recall and F-measure
precision = precision_score(
    actual_labels,
    predicted_labels,
    average='macro',
    zero_division=0
)

recall = recall_score(
    actual_labels,
    predicted_labels,
    average='macro',
    zero_division=0
)

f_measure = f1_score(
    actual_labels,
    predicted_labels,
    average='macro',
    zero_division=0
)

# Step 7: Display results
print("Number of Documents:", len(documents))
print("Number of Features:", X.shape[1])
print("Number of Clusters:", k)

print("\nPerformance Measures:")
print("Purity    :", round(purity, 4))
print("Precision :", round(precision, 4))
print("Recall    :", round(recall, 4))
print("F-measure :", round(f_measure, 4))

Number of Documents: 1657
Number of Features: 5000
Number of Clusters: 3

Performance Measures:
Purity    : 0.717
Precision : 0.808
Recall    : 0.7346
F-measure : 0.7145
